# CPV Dataset Inspector

Interactive notebook for exploring CPV-schema datasets. Set `SOURCE` to a local `.parquet` file or a HuggingFace dataset ID.

In [ ]:
# --- Configuration ---
SOURCE = "tests/medqa_cpv/medqa_cpv.parquet"  # or e.g. "kenza-ily/medqa-cpv"
CONFIG = None   # HuggingFace config name, if needed
SPLIT  = "train"  # HuggingFace split, if loading from Hub

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

repo_root = str(Path("__file__").resolve().parent.parent)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from tests.utils import (
    validate_cpv_schema,
    check_demographic_distribution,
    CPV_REQUIRED_COLUMNS,
)
from explore.dataset_stats import load_source

df = load_source(SOURCE, config=CONFIG, split=SPLIT)
print(f"Loaded {len(df):,} rows from: {SOURCE}")

In [ ]:
# Shape + dtypes
print(f"Shape: {df.shape}")
df.dtypes

In [ ]:
# Null counts
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]
if null_counts.empty:
    print("No null values.")
else:
    display(null_counts.to_frame("null_count"))

In [ ]:
# Schema validation
allow_null_d = "option_d" in df.columns and df["option_d"].isna().all()
errors = validate_cpv_schema(df, allow_null_option_d=allow_null_d)
if not errors:
    print("Schema: PASS")
else:
    for e in errors:
        print(f"FAIL: {e}")

In [ ]:
# Demographic crosstab
dist = check_demographic_distribution(df)
print(f"All 10 variants per case: {dist['all_10_variants']}")
display(dist["crosstab"])

In [ ]:
# Answer label distribution
if "answer_idx" in df.columns:
    counts = df["answer_idx"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(5, 3))
    counts.plot.bar(ax=ax, color="steelblue", edgecolor="black")
    ax.set_title("Answer Label Distribution")
    ax.set_xlabel("Answer")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()
    display(counts.to_frame("count"))

In [ ]:
# Specialty distribution (if present) + sample rows
if "specialty" in df.columns:
    top20 = df["specialty"].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(8, 5))
    top20.plot.barh(ax=ax, color="coral", edgecolor="black")
    ax.set_title("Top 20 Specialties")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

present_cols = [c for c in CPV_REQUIRED_COLUMNS if c in df.columns]
print("\nSample rows (required columns):")
with pd.option_context("display.max_colwidth", 60):
    display(df[present_cols].head(5))